# Домашнее задание № 8

### Задание 1 (7 баллов).
Это задание основано на этой тетрадке - https://github.com/mannefedov/compling_nlp_hse_course/blob/master/notebooks/transfer_learning_hg/Fine_tunining_pretrained_LMs_torch.ipynb

На датасете lenta_sample.ru  дообучите две модели - modernbert-base (из семинара) и rumodernbert-base (https://huggingface.co/deepvk/RuModernBERT-base). Оцените разницу в качестве сравнив поклассовые метрики (classification_report)

Для обоих моделей качество должно быть >0.10 по f-мере (прогоните несколько экспериментов если у вас получаются нули, изменяя параметры).
Также для обоих моделей попробуйте дообучать модель и целиком и дообучать только последний слой.
Для RuModernBERT дополнительно сравните модель, которая использует первый вектор (cls токен, как в семинаре), так и усредненный вектор по всем hidden_state, который выдает bert.


### Подзадание на 1 балл
Дообучите любую из моделей, добавляя к BERT основе не только один линейный слой, но и один слой LSTM. Финальное состояние из этого слоя должно подаваться с новый линейный классификационный слой.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv(
    "C:/Users/Ale03x/Downloads/lenta_sample.csv",
    sep=",",
    encoding="utf-8",
    quotechar='"'
)

df = df.dropna(subset=["text", "topic"])

df["full_text"] = df["title"].fillna("") + " " + df["text"]

counts = df["topic"].value_counts()
valid_topics = counts[counts > 5].index
df = df[df["topic"].isin(valid_topics)]

print("Размер датасета:", df.shape)
print(df["topic"].value_counts())


Размер датасета: (607, 7)
topic
Из жизни             55
Бывший СССР          54
Наука и техника      54
Культура             53
Ценности             45
Дом                  45
Интернет и СМИ       44
Бизнес               44
Силовые структуры    40
Спорт                39
Россия               32
Экономика            32
Мир                  27
69-я параллель       13
Легпром              13
Библиотека           10
Крым                  7
Name: count, dtype: int64


In [3]:
texts = df["full_text"].values
labels = df["topic"].values

label2id = {label: i for i, label in enumerate(sorted(set(labels)))}
id2label = {v: k for k, v in label2id.items()}

y = np.array([label2id[l] for l in labels])

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

In [5]:
class BertClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        pooling="cls",
        freeze_bert=False,
        use_lstm=False,
        lstm_hidden=256
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        self.pooling = pooling
        self.use_lstm = use_lstm

        hidden_size = self.bert.config.hidden_size

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        if use_lstm:
            self.lstm = nn.LSTM(
                hidden_size,
                lstm_hidden,
                batch_first=True
            )
            self.classifier = nn.Linear(lstm_hidden, num_classes)
        else:
            self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden = outputs.last_hidden_state

        if self.use_lstm:
            _, (h_n, _) = self.lstm(last_hidden)
            pooled = h_n[-1]
        else:
            if self.pooling == "cls":
                pooled = last_hidden[:, 0]
            elif self.pooling == "mean":
                mask = attention_mask.unsqueeze(-1)
                pooled = (last_hidden * mask).sum(1) / mask.sum(1)

        logits = self.classifier(pooled)
        return logits


In [6]:
def train_model(model, train_loader, test_loader, epochs=3, lr=2e-5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

    evaluate_model(model, test_loader)

In [7]:
def evaluate_model(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    preds = []
    true = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            predictions = torch.argmax(outputs, dim=1)

            preds.extend(predictions.cpu().numpy())
            true.extend(labels.cpu().numpy())

    print(classification_report(true, preds, target_names=id2label.values()))

In [8]:
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = TextDataset(X_train, y_train, tokenizer)
test_ds = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

model = BertClassifier(
    model_name,
    num_classes=len(label2id),
    pooling="cls",
    freeze_bert=False
)

train_model(model, train_loader, test_loader)

100%|██████████| 31/31 [01:58<00:00,  3.83s/it]


Epoch 1 | Loss: 2.7995


100%|██████████| 31/31 [01:58<00:00,  3.82s/it]


Epoch 2 | Loss: 2.6871


100%|██████████| 31/31 [01:59<00:00,  3.87s/it]


Epoch 3 | Loss: 2.5629
                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00         3
       Библиотека       1.00      1.00      1.00         2
           Бизнес       1.00      0.22      0.36         9
      Бывший СССР       0.14      0.27      0.19        11
              Дом       0.00      0.00      0.00         9
         Из жизни       0.13      0.64      0.22        11
   Интернет и СМИ       0.12      0.22      0.15         9
             Крым       0.00      0.00      0.00         1
         Культура       0.00      0.00      0.00        11
          Легпром       0.00      0.00      0.00         3
              Мир       0.00      0.00      0.00         5
  Наука и техника       0.16      0.36      0.22        11
           Россия       0.00      0.00      0.00         6
Силовые структуры       0.00      0.00      0.00         8
            Спорт       0.00      0.00      0.00         8
         Ценности       0.00    

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

In [9]:
model = BertClassifier(
    model_name,
    num_classes=len(label2id),
    pooling="cls",
    freeze_bert=True
)

train_model(model, train_loader, test_loader, lr=1e-3)


100%|██████████| 31/31 [00:40<00:00,  1.29s/it]


Epoch 1 | Loss: 2.8766


100%|██████████| 31/31 [00:38<00:00,  1.25s/it]


Epoch 2 | Loss: 2.6151


100%|██████████| 31/31 [00:38<00:00,  1.24s/it]


Epoch 3 | Loss: 2.4323
                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00         3
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.40      0.22      0.29         9
      Бывший СССР       0.25      0.18      0.21        11
              Дом       0.33      0.11      0.17         9
         Из жизни       0.12      0.45      0.19        11
   Интернет и СМИ       0.06      0.11      0.08         9
             Крым       0.00      0.00      0.00         1
         Культура       0.05      0.09      0.06        11
          Легпром       0.50      0.33      0.40         3
              Мир       0.00      0.00      0.00         5
  Наука и техника       0.00      0.00      0.00        11
           Россия       0.00      0.00      0.00         6
Силовые структуры       0.50      0.12      0.20         8
            Спорт       0.00      0.00      0.00         8
         Ценности       0.26    

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

In [10]:
model_name = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = TextDataset(X_train, y_train, tokenizer)
test_ds = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

model = BertClassifier(
    model_name,
    num_classes=len(label2id),
    pooling="cls",
    freeze_bert=False
)

train_model(model, train_loader, test_loader)

100%|██████████| 31/31 [01:59<00:00,  3.85s/it]


Epoch 1 | Loss: 2.8071


100%|██████████| 31/31 [01:58<00:00,  3.82s/it]


Epoch 2 | Loss: 1.7705


100%|██████████| 31/31 [01:59<00:00,  3.86s/it]


Epoch 3 | Loss: 0.7394
                   precision    recall  f1-score   support

   69-я параллель       1.00      0.33      0.50         3
       Библиотека       1.00      1.00      1.00         2
           Бизнес       0.45      0.56      0.50         9
      Бывший СССР       0.71      0.45      0.56        11
              Дом       0.82      1.00      0.90         9
         Из жизни       0.57      0.73      0.64        11
   Интернет и СМИ       0.21      0.33      0.26         9
             Крым       0.00      0.00      0.00         1
         Культура       0.58      0.64      0.61        11
          Легпром       0.00      0.00      0.00         3
              Мир       0.14      0.20      0.17         5
  Наука и техника       0.86      0.55      0.67        11
           Россия       0.40      0.67      0.50         6
Силовые структуры       0.50      0.38      0.43         8
            Спорт       1.00      0.62      0.77         8
         Ценности       0.64    

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

In [11]:
model = BertClassifier(
    model_name,
    num_classes=len(label2id),
    pooling="mean",
    freeze_bert=False
)

train_model(model, train_loader, test_loader)

100%|██████████| 31/31 [02:00<00:00,  3.90s/it]


Epoch 1 | Loss: 2.5980


100%|██████████| 31/31 [02:00<00:00,  3.89s/it]


Epoch 2 | Loss: 1.4456


100%|██████████| 31/31 [01:59<00:00,  3.87s/it]


Epoch 3 | Loss: 0.6912
                   precision    recall  f1-score   support

   69-я параллель       1.00      0.33      0.50         3
       Библиотека       1.00      0.50      0.67         2
           Бизнес       0.43      1.00      0.60         9
      Бывший СССР       0.58      0.64      0.61        11
              Дом       0.89      0.89      0.89         9
         Из жизни       0.83      0.45      0.59        11
   Интернет и СМИ       0.55      0.67      0.60         9
             Крым       0.00      0.00      0.00         1
         Культура       0.60      0.82      0.69        11
          Легпром       0.00      0.00      0.00         3
              Мир       0.00      0.00      0.00         5
  Наука и техника       0.92      1.00      0.96        11
           Россия       0.00      0.00      0.00         6
Силовые структуры       0.43      0.75      0.55         8
            Спорт       0.80      1.00      0.89         8
         Ценности       1.00    

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

In [12]:
model = BertClassifier(
    model_name,
    num_classes=len(label2id),
    pooling="cls",
    freeze_bert=False,
    use_lstm=True
)

train_model(model, train_loader, test_loader)


100%|██████████| 31/31 [01:49<00:00,  3.54s/it]


Epoch 1 | Loss: 2.8302


100%|██████████| 31/31 [01:57<00:00,  3.79s/it]


Epoch 2 | Loss: 2.6732


100%|██████████| 31/31 [02:00<00:00,  3.89s/it]


Epoch 3 | Loss: 2.5482
                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00         3
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.00      0.00      0.00         9
      Бывший СССР       0.17      0.73      0.27        11
              Дом       1.00      0.11      0.20         9
         Из жизни       0.30      0.27      0.29        11
   Интернет и СМИ       0.00      0.00      0.00         9
             Крым       0.00      0.00      0.00         1
         Культура       0.12      0.27      0.17        11
          Легпром       0.00      0.00      0.00         3
              Мир       0.00      0.00      0.00         5
  Наука и техника       0.36      0.36      0.36        11
           Россия       0.00      0.00      0.00         6
Силовые структуры       0.00      0.00      0.00         8
            Спорт       0.00      0.00      0.00         8
         Ценности       0.28    

C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ale03x\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

RuModernBERT лучше ModernBERT на русском, что логично, ведь датасет полностью на русском языке, усреднение токенов (mean pooling) даёт небольшой выигрыш по F1.

Добавление LSTM ничего существенного не добавило и не улучшило.

## Задание 2 (2 балла)

Проведите несколько экспериментов с gradual unfreezing. Используя ту же задачу и датасет, что в первом задании, дообучите модели таким образом, что в начале обучения весь backbone BERT заморожен и дообучение происходит только на новом Linear/dense/fc слое, но после каждой N эпохи, размораживайте (`_requires_grad = True`) какую-то часть параметров BERT. Проведите как минимум 3 эксперимента с разными методиками разморозки. Вы можете придумать их сами, но вот несколько примеров - 1) разморозка начинается с конца модели и каждые две эпохи размораживается несколько слоев до тех пор пока не разморозятся все слои всключая первый эмбеддинг слой 2) разморозка начинается с начала (то есть сначала размораживается только embedding слой), а со временем включаются и все последующие 3) таргетировано размораживаются сначала слои query трансформаций, потом key, потом value 4) или даже можете попробовать сначала разморозить все слои и наоборот замораживать части с прогрессом обучения. Для каждого эксперимента сохраните метрики на каждой эпохе и финальную метрику, сравните их между собой - какой schedule сработал лучше всего? Сработало ли что-то лучше дефолтного подхода из первого задания?


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

class BertClassifier(nn.Module):
    def __init__(self, model_name, num_classes, pooling="cls"):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.pooling = pooling
        hidden_size = self.bert.config.hidden_size
        
        self.classifier = nn.Linear(hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        if self.pooling == "cls":
            pooled = outputs.last_hidden_state[:, 0]  # CLS токен
        elif self.pooling == "mean":
            pooled = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)

def train_with_schedule(model, train_loader, test_loader, schedule, epochs_per_step=2, lr=2e-5):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    history = {"train_loss": [], "test_f1": []}
    
    total_epochs = sum([step['epochs'] for step in schedule])
    current_step = 0
    
    for step in schedule:

        for name, param in model.bert.named_parameters():
            param.requires_grad = name in step['unfreeze']
        
        for epoch in range(step['epochs']):
            model.train()
            total_loss = 0
            for batch in train_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                optimizer.zero_grad()
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / len(train_loader)
            history["train_loss"].append(avg_loss)
            
            model.eval()
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for batch in test_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    
                    outputs = model(input_ids, attention_mask)
                    preds = torch.argmax(outputs, dim=1)
                    
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
            
            report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
            f1_macro = report['macro avg']['f1-score']
            history["test_f1"].append(f1_macro)
            
            print(f"Step {current_step+1}/{len(schedule)}, Epoch {epoch+1}/{step['epochs']}, "
                  f"Train Loss: {avg_loss:.4f}, Test Macro F1: {f1_macro:.4f}")
        
        current_step += 1
    
    return history

all_layers = [f'encoder.layer.{i}.' for i in range(12)] + ['embeddings.']

Using device: cpu


In [24]:
schedule_end_to_start = [
    {"epochs": 2, "unfreeze": []}, 
    {"epochs": 2, "unfreeze": all_layers[9:]},  
    {"epochs": 2, "unfreeze": all_layers[6:]},
    {"epochs": 2, "unfreeze": all_layers} 
]


schedule_start_to_end = [
    {"epochs": 2, "unfreeze": ['embeddings.']},
    {"epochs": 2, "unfreeze": all_layers[:4] + ['embeddings.']},
    {"epochs": 2, "unfreeze": all_layers[:8] + ['embeddings.']},
    {"epochs": 2, "unfreeze": all_layers + ['embeddings.']}
]


qkv_layers = [f'encoder.layer.{i}.attention.self.query.' for i in range(12)]
k_layers = [f'encoder.layer.{i}.attention.self.key.' for i in range(12)]
v_layers = [f'encoder.layer.{i}.attention.self.value.' for i in range(12)]
schedule_qkv = [
    {"epochs": 2, "unfreeze": []},
    {"epochs": 2, "unfreeze": qkv_layers},
    {"epochs": 2, "unfreeze": qkv_layers + k_layers},
    {"epochs": 2, "unfreeze": qkv_layers + k_layers + v_layers}
]

model1 = BertClassifier(model_name, len(label2id)).to(device)
hist1 = train_with_schedule(model1, train_loader, test_loader, schedule_end_to_start)

model2 = BertClassifier(model_name, len(label2id)).to(device)
hist2 = train_with_schedule(model2, train_loader, test_loader, schedule_start_to_end)

model3 = BertClassifier(model_name, len(label2id)).to(device)
hist3 = train_with_schedule(model3, train_loader, test_loader, schedule_qkv)

print("Experiment 1 final F1:", hist1['test_f1'][-1])
print("Experiment 2 final F1:", hist2['test_f1'][-1])
print("Experiment 3 final F1:", hist3['test_f1'][-1])


Step 1/4, Epoch 1/2, Train Loss: 2.9398, Test Macro F1: 0.0097
Step 1/4, Epoch 2/2, Train Loss: 2.9260, Test Macro F1: 0.0097
Step 2/4, Epoch 1/2, Train Loss: 2.8991, Test Macro F1: 0.0097
Step 2/4, Epoch 2/2, Train Loss: 2.8840, Test Macro F1: 0.0097
Step 3/4, Epoch 1/2, Train Loss: 2.8688, Test Macro F1: 0.0097
Step 3/4, Epoch 2/2, Train Loss: 2.8501, Test Macro F1: 0.0097
Step 4/4, Epoch 1/2, Train Loss: 2.8400, Test Macro F1: 0.0097
Step 4/4, Epoch 2/2, Train Loss: 2.8286, Test Macro F1: 0.0097
Step 1/4, Epoch 1/2, Train Loss: 2.8837, Test Macro F1: 0.0019
Step 1/4, Epoch 2/2, Train Loss: 2.8692, Test Macro F1: 0.0019
Step 2/4, Epoch 1/2, Train Loss: 2.8534, Test Macro F1: 0.0019
Step 2/4, Epoch 2/2, Train Loss: 2.8429, Test Macro F1: 0.0019
Step 3/4, Epoch 1/2, Train Loss: 2.8294, Test Macro F1: 0.0019
Step 3/4, Epoch 2/2, Train Loss: 2.8190, Test Macro F1: 0.0019
Step 4/4, Epoch 1/2, Train Loss: 2.8071, Test Macro F1: 0.0020
Step 4/4, Epoch 2/2, Train Loss: 2.8023, Test Macro F1:

F1 низкий во всех экспериментах, Train Loss падает медленно, значит, модель учится, но оооочень медленно.